<a href="https://colab.research.google.com/github/chakkeiwong/agents/blob/master/QZ.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import tensorflow as tf

# ──────────────────────────────────────────────────────────────────────────────
#  Utility: RQ decomposition  M = R @ Q  (upper-triangular R, unitary Q)
#  Uses np.linalg.qr internally — no SciPy required.
# ──────────────────────────────────────────────────────────────────────────────
def _rq(M):
    """
    RQ decomposition via the flip-conjugate-transpose trick:
        1. Form  M' = J @ conj(M).T @ J  (J = anti-diagonal identity)
        2. QR of M' gives Q_f, R_f
        3. R = J @ conj(R_f).T @ J  (upper triangular)
           Q = J @ conj(Q_f).T @ J  (unitary)
    Then  M = R @ Q.
    """
    m = M.shape[0]
    J = np.eye(m, dtype=M.dtype)[::-1]           # anti-diagonal identity
    Q_f, R_f = np.linalg.qr(J @ M.conj().T @ J)  # QR of flipped conj-transpose
    R = J @ R_f.conj().T @ J                      # upper triangular
    Q = J @ Q_f.conj().T @ J                      # unitary
    return R, Q


# ──────────────────────────────────────────────────────────────────────────────
#  Utility: Givens rotation coefficients
# ──────────────────────────────────────────────────────────────────────────────
def _givens(a, b):
    """
    Return (c, s) with c real ≥ 0, |c|²+|s|² = 1, such that:
        [[c,  s ],
         [-s*, c]] @ [a; b]  =  [r; 0],   r = hypot(|a|, |b|).
    """
    r = np.hypot(abs(a), abs(b))
    if r < 1e-15:
        return 1.0, 0.0 + 0j
    c = abs(a) / r
    s = ((a / abs(a)) * np.conj(b) / r) if abs(a) > 0 else complex(b / r)
    return float(c), complex(s)


# ──────────────────────────────────────────────────────────────────────────────
#  Apply Givens from the LEFT  (rows i and j)
# ──────────────────────────────────────────────────────────────────────────────
def _rot_left(M, c, s, i, j):
    ri, rj = M[i].copy(), M[j].copy()
    M[i] =  c * ri + s * rj
    M[j] = -np.conj(s) * ri + c * rj


# ──────────────────────────────────────────────────────────────────────────────
#  Apply Givens from the RIGHT  (cols i and j)
# ──────────────────────────────────────────────────────────────────────────────
def _rot_right(M, c, s, i, j):
    ci, cj = M[:, i].copy(), M[:, j].copy()
    M[:, i] =  c * ci - np.conj(s) * cj
    M[:, j] =  s * ci + c * cj


# ══════════════════════════════════════════════════════════════════════════════
#  PHASE 1 — Hessenberg-Triangular Reduction
# ══════════════════════════════════════════════════════════════════════════════
def _hess_tri(A, B):
    """
    Reduce (A, B) so that A is upper-Hessenberg and B is upper-triangular.

    Algorithm:
      1. QR(B) makes B upper-triangular; apply Q₀ᴴ from the left to A.
      2. For k = 0 … n-3:
           a. Left Householder to zero A[k+2:, k]  → creates fill-in in B.
           b. RQ of B[k+1:, k+1:] restores B to upper-triangular
              by a right-unitary multiplication (which preserves the zeros
              already established in A).

    Returns H, T, Q, Z  with  A = Q @ H @ Zᴴ,  B = Q @ T @ Zᴴ.
    """
    n = A.shape[0]
    A = A.astype(np.complex128)
    B = B.astype(np.complex128)

    # ── Step 1: QR(B) ─────────────────────────────────────────────────────────
    Q0, R = np.linalg.qr(B)     # B = Q0 @ R
    A  = Q0.conj().T @ A        # A ← Q0ᴴ A
    B  = R.copy()               # B ← R  (now upper triangular)
    Q  = Q0.copy()              # cumulative left unitary
    Z  = np.eye(n, dtype=np.complex128)   # cumulative right unitary

    # ── Step 2: Hessenberg reduction ─────────────────────────────────────────
    for k in range(n - 2):

        # -- Householder from LEFT to zero A[k+2:, k] -------------------------
        x   = A[k+1:, k]
        nrm = np.linalg.norm(x)
        if nrm < 1e-15:
            continue

        alpha = -(np.sign(x[0]) if abs(x[0]) > 0 else 1.0) * nrm
        u = x.copy()
        u[0] -= alpha
        u /= np.linalg.norm(u)          # unit Householder vector

        # Apply  H_tilde = I − 2 u uᴴ  from the left
        A[k+1:, :] -= 2.0 * np.outer(u, u.conj() @ A[k+1:, :])
        B[k+1:, :] -= 2.0 * np.outer(u, u.conj() @ B[k+1:, :])

        # Update Q: Q_new = Q_old @ (I ⊕ H_tilde)
        Q[:, k+1:] -= 2.0 * np.outer(Q[:, k+1:] @ u, u.conj())

        # -- RQ of B[k+1:, k+1:] → restores B to upper-triangular -------------
        R_sub, Q_rq = _rq(B[k+1:, k+1:])
        Qh = Q_rq.conj().T              # = Q_rq⁻¹

        B[k+1:, k+1:] = R_sub          # replace sub-block with upper-tri R
        B[:k+1, k+1:] = B[:k+1, k+1:] @ Qh
        A[:,    k+1:] = A[:,    k+1:] @ Qh

        # Update Z: Z_new = Z_old @ Q_rqᴴ
        Z[:, k+1:] = Z[:, k+1:] @ Qh

    return A, B, Q, Z


# ══════════════════════════════════════════════════════════════════════════════
#  PHASE 2 — QZ Iteration (Francis implicit single-shift)
# ══════════════════════════════════════════════════════════════════════════════
def _wilkinson_shift(H, T, hi):
    """
    Generalized Wilkinson shift from the 2×2 trailing sub-problem.
    Returns the eigenvalue closer to H[hi-1,hi-1] / T[hi-1,hi-1].
    """
    a11, a12 = H[hi-2, hi-2], H[hi-2, hi-1]
    a21, a22 = H[hi-1, hi-2], H[hi-1, hi-1]
    b11, b22 = T[hi-2, hi-2], T[hi-1, hi-1]

    if abs(b11) < 1e-15 or abs(b22) < 1e-15:
        return a22   # fallback

    t11 = a11 / b11
    t12 = (a12 - T[hi-2, hi-1] * a22 / b22) / b11
    t22 = a22 / b22

    tr   = t11 + t22
    det  = t11 * t22 - t12 * a21
    disc = tr * tr - 4.0 * det
    sq   = np.sqrt(complex(disc))
    s1, s2 = (tr + sq) / 2.0, (tr - sq) / 2.0
    return s1 if abs(s1 - t22) <= abs(s2 - t22) else s2


def _qz_step(H, T, Q, Z, lo, hi):
    """
    One implicit single-shift QZ step on the active block [lo : hi].

    Each sub-step j:
      1. Left  Givens G_L (rows j, j+1) — introduces / chases the bulge in H.
      2. Right Givens G_R (cols j, j+1) — zeros T[j+1, j] created by G_L.

    Accumulators are updated so that  A = Q H Zᴴ,  B = Q T Zᴴ  remains exact.
    """
    mu = _wilkinson_shift(H, T, hi)

    # Initial (f, g) define the first Givens (implicit shift step)
    f = H[lo, lo] - mu * T[lo, lo]
    g = H[lo + 1, lo]               # T[lo+1, lo] = 0 (T is upper-tri)

    for j in range(lo, hi - 1):

        # ── Left Givens G_L ───────────────────────────────────────────────────
        r = np.hypot(abs(f), abs(g))
        if r < 1e-15:
            if j + 2 < hi:
                f, g = H[j + 1, j], H[j + 2, j]
            continue

        c_L = abs(f) / r
        s_L = ((f / abs(f)) * np.conj(g) / r) if abs(f) > 0 else complex(g / r)

        _rot_left(H, c_L, s_L, j, j + 1)
        _rot_left(T, c_L, s_L, j, j + 1)

        # Q ← Q G_Lᴴ   (G_Lᴴ = [[c_L, −s_L], [s_L*, c_L]])
        cj, cj1 = Q[:, j].copy(), Q[:, j + 1].copy()
        Q[:, j]     =  c_L * cj + np.conj(s_L) * cj1
        Q[:, j + 1] = -s_L * cj + c_L * cj1

        # ── Right Givens G_R — zeros T[j+1, j] ────────────────────────────────
        # After G_L: T[j+1, j] = −s_L* · T_old[j,j]  (fill-in).
        # Pivot is T[j+1, j+1]; compute (c_R, s_R) to annihilate T[j+1, j].
        piv = T[j + 1, j + 1]
        sub = T[j + 1, j]
        r_R = np.hypot(abs(piv), abs(sub))

        if r_R < 1e-15:
            if j + 2 < hi:
                f, g = H[j + 1, j], H[j + 2, j]
            continue

        c_R = abs(piv) / r_R
        s_R = ((piv / abs(piv)) * np.conj(sub) / r_R) if abs(piv) > 0 else complex(sub / r_R)

        _rot_right(H, c_R, s_R, j, j + 1)
        _rot_right(T, c_R, s_R, j, j + 1)

        # Z ← Z G_R   (G_R = [[c_R, −s_R*], [s_R, c_R]])
        cj, cj1 = Z[:, j].copy(), Z[:, j + 1].copy()
        Z[:, j]     =  c_R * cj - np.conj(s_R) * cj1
        Z[:, j + 1] =  s_R * cj + c_R * cj1

        # Next (f, g) for the bulge-chasing step
        f = H[j + 1, j]
        g = H[j + 2, j] if (j + 2 < hi) else 0.0

    return H, T, Q, Z


def _qz_iters(H, T, Q, Z, max_sweeps=300, tol=1e-12):
    """
    Deflating QZ iterations.

    Repeatedly applies _qz_step to the bottom-most un-converged sub-problem
    until all sub-diagonal entries of H are negligible (upper-triangular form).
    """
    n  = H.shape[0]
    hi = n

    for _ in range(max_sweeps * n):
        # ── Deflation: shrink hi while H[hi-1, hi-2] is negligible ────────────
        while hi > 1:
            thresh = tol * (abs(H[hi - 2, hi - 2]) + abs(H[hi - 1, hi - 1]))
            if abs(H[hi - 1, hi - 2]) <= thresh:
                H[hi - 1, hi - 2] = 0.0
                hi -= 1
            else:
                break

        if hi <= 1:
            break   # fully converged

        # ── Find start of active sub-problem ──────────────────────────────────
        lo = hi - 1
        while lo > 0:
            thresh = tol * (abs(H[lo - 1, lo - 1]) + abs(H[lo, lo]))
            if abs(H[lo, lo - 1]) <= thresh:
                H[lo, lo - 1] = 0.0
                break
            lo -= 1

        H, T, Q, Z = _qz_step(H, T, Q, Z, lo, hi)

    return H, T, Q, Z


# ══════════════════════════════════════════════════════════════════════════════
#  Public API — qz_decomposition
# ══════════════════════════════════════════════════════════════════════════════
def qz_decomposition(A, B):
    """
    Generalized Schur (QZ) Decomposition — TensorFlow interface, no SciPy.

    Finds unitary Q, Z and upper-triangular AA, BB such that:

        A  =  Q @ AA @ Zᴴ
        B  =  Q @ BB @ Zᴴ

    Generalized eigenvalues:  λᵢ = diag(AA)[i] / diag(BB)[i]

    Parameters
    ----------
    A : tf.Tensor, shape (n, n)
    B : tf.Tensor, shape (n, n)

    Returns
    -------
    AA : tf.Tensor  — upper-triangular Schur form of A
    BB : tf.Tensor  — upper-triangular Schur form of B
    Q  : tf.Tensor  — left  unitary transformation  (Q @ AA @ Zᴴ = A)
    Z  : tf.Tensor  — right unitary transformation  (Q @ BB @ Zᴴ = B)
    """
    cdtype = tf.complex128
    A_np = tf.cast(A, cdtype).numpy()
    B_np = tf.cast(B, cdtype).numpy()

    # Phase 1 — Hessenberg-triangular form
    H, T, Q, Z = _hess_tri(A_np, B_np)

    # Phase 2 — QZ iterations → upper-triangular form
    AA_np, BB_np, Q_np, Z_np = _qz_iters(H, T, Q, Z)

    return (
        tf.constant(AA_np, dtype=cdtype),
        tf.constant(BB_np, dtype=cdtype),
        tf.constant(Q_np,  dtype=cdtype),
        tf.constant(Z_np,  dtype=cdtype),
    )


# ══════════════════════════════════════════════════════════════════════════════
#  Demo and Verification
# ══════════════════════════════════════════════════════════════════════════════
def testcode():
    np.random.seed(0)
    n = 6

    # Random complex matrix pair
    rng = np.random.default_rng(42)
    A_np = rng.standard_normal((n, n)) + 1j * rng.standard_normal((n, n))
    B_np = rng.standard_normal((n, n)) + 1j * rng.standard_normal((n, n))

    A = tf.constant(A_np, dtype=tf.complex128)
    B = tf.constant(B_np, dtype=tf.complex128)

    # ── Run QZ ────────────────────────────────────────────────────────────────
    AA, BB, Q, Z = qz_decomposition(A, B)

    # ── Reconstruction errors ─────────────────────────────────────────────────
    Zh = tf.linalg.adjoint(Z)
    A_rec = Q @ AA @ Zh
    B_rec = Q @ BB @ Zh

    err_A = tf.reduce_max(tf.abs(A - A_rec)).numpy()
    err_B = tf.reduce_max(tf.abs(B - B_rec)).numpy()

    I = tf.eye(n, dtype=tf.complex128)
    err_Q = tf.reduce_max(tf.abs(Q @ tf.linalg.adjoint(Q) - I)).numpy()
    err_Z = tf.reduce_max(tf.abs(Z @ tf.linalg.adjoint(Z) - I)).numpy()

    print("=" * 55)
    print("  QZ Decomposition Verification")
    print("=" * 55)
    print(f"  max |A − Q·AA·Zᴴ|  = {err_A:.2e}")
    print(f"  max |B − Q·BB·Zᴴ|  = {err_B:.2e}")
    print(f"  Q unitarity error  = {err_Q:.2e}")
    print(f"  Z unitarity error  = {err_Z:.2e}")

    # ── Generalized eigenvalues ───────────────────────────────────────────────
    aa_diag = tf.linalg.diag_part(AA).numpy()
    bb_diag = tf.linalg.diag_part(BB).numpy()
    eigs    = aa_diag / bb_diag

    print("\n  Generalized eigenvalues  λ = diag(AA) / diag(BB):")
    for i, lam in enumerate(eigs):
        print(f"    λ_{i} = {lam.real:+.6f}  {lam.imag:+.6f}j")
    print("=" * 55)

In [ ]:
testcode()

  QZ Decomposition Verification
  max |A − Q·AA·Zᴴ|  = 2.13e-13
  max |B − Q·BB·Zᴴ|  = 3.57e-15
  Q unitarity error  = 1.11e-15
  Z unitarity error  = 6.66e-16

  Generalized eigenvalues  λ = diag(AA) / diag(BB):
    λ_0 = -2.362069  -3.372762j
    λ_1 = +0.551182  +1.145291j
    λ_2 = +0.762150  +0.302203j
    λ_3 = -0.151521  +0.247414j
    λ_4 = -0.399882  -0.603570j
    λ_5 = -0.016842  -0.304488j
